# OpenEO Process graphs in JupyterGIS

This Notebook creates of OpenEO process graphs, and makes use of a local [titiler-openeo](https://sentinel-hub.github.io/titiler-openeo) server to serve them to JupyterGIS as tile layers on port `8080`.

Follow this guide to spawn a server locally on your machine so that the notebook can run https://sentinel-hub.github.io/titiler-openeo/local-setup/#environment-setup

Alternatively, one can change the following cell and provide a URL to another `titiler-openeo` server.

⚠️ This Notebook cannot run in a JupyterLite/Notebook.link context unless using a remote `titiler-openeo` server

In [ ]:
SERVER_URL = "http://127.0.0.1:8080/"

BASIC_AUTH = "test"

In [ ]:
import openeo
from openeo.processes import process
from IPython.display import Image
from jupytergis import GISDocument

connection = openeo.connect(SERVER_URL)

In [ ]:
connection.authenticate_basic(username=BASIC_AUTH, password=BASIC_AUTH)

In [ ]:
# connection.auth.bearer

In [ ]:
# connection.list_collection_ids()

In [ ]:
collection_info = connection.describe_collection("sentinel-2-global-mosaics")

In [ ]:
collection_info

In [ ]:
load1 = connection.datacube_from_process(
    "load_collection",
    id="sentinel-2-global-mosaics",
    bands=["B04", "B03", "B02"],
    properties={},
    # Spatial extent is required for OpenEO but since we're viewing a "Global mosaic" with titiler-openeo, it has no impact on the result
    spatial_extent={},
    temporal_extent=["2022-04-15T00:00:00Z", "2022-12-31T00:00:00Z"],
)
load1 = load1.reduce_dimension(
    dimension="time",
    reducer="first",
)

In [ ]:
def process1(x, context=None):
    data1 = process(
        "linear_scale_range", inputMax=10000, inputMin=0, outputMax=255, x=x
    )
    data2 = process("trunc", x=data1)
    return data2

In [ ]:
processed = load1.apply(process=process1)

color = processed.process(
    "color_formula",
    data=processed,
    formula="Gamma RGB 1.5 Sigmoidal RGB 6 0.3 Saturation 1",
)

In [ ]:
color

In [ ]:
color = color.save_result(format="PNG")

doc = GISDocument(latitude=40.7128, longitude=74.0060, zoom=12)
doc.add_raster_layer(url="https://basemap.nationalmap.gov/arcgis/rest/services/USGSTopo/MapServer/tile/{z}/{y}/{x}")
doc.add_openeo_tile_layer(color)
doc

In [ ]:
spatial_extent_east = -73.90
spatial_extent_north = 40.80
spatial_extent_south = 40.70
spatial_extent_west = -74.00


cube = connection.load_collection(
    "sentinel-2-global-mosaics",
    bands=["B03", "B08"],
    spatial_extent={
        "east": spatial_extent_east,
        "north": spatial_extent_north,
        "south": spatial_extent_south,
        "west": spatial_extent_west,
    },
    temporal_extent=["2022-04-15", "2022-12-31"],
)

cube = cube.reduce_dimension(
    dimension="t",
    reducer="first",
)

cube = cube / 10000.0

# NDWI = (GREEN - NIR) / (GREEN + NIR)
ndwi = cube.ndvi(nir=0, red=1)

ndwi_vis = (ndwi + 1) / 2

ndwi_png = ndwi_vis.linear_scale_range(
    input_min=0,
    input_max=1,
    output_min=0,
    output_max=255,
)

result = ndwi_png.save_result(format="PNG")

doc = GISDocument(latitude=40.75, longitude=-73.95, zoom=12)
doc.add_raster_layer(url='https://basemap.nationalmap.gov/arcgis/rest/services/USGSTopo/MapServer/tile/{z}/{y}/{x}")
doc.add_openeo_tile_layer(result)
doc

In [ ]:
cube = connection.load_collection( 
    "sentinel-2-global-mosaics", 
    bands=["B08", "B04"], 
    spatial_extent={ 
        "east": spatial_extent_east, 
        "north": spatial_extent_north, 
        "south": spatial_extent_south, 
        "west": spatial_extent_west, 
    }, 
    temporal_extent=["2022-04-15", "2022-12-31"], 
) 

cube = cube.reduce_dimension(dimension="t", reducer="first")
ndvi = cube.ndvi(nir=0, red=1) 

# Scale NDVI from [-1, 1] -> [0, 255]
ndvi_vis = (
    ndvi + 1
) * 127.5

ndvi_png = ndvi_vis.linear_scale_range(
    input_min=0,
    input_max=255,
    output_min=0,
    output_max=255,
)

result = ndvi_png.save_result(format="PNG")

doc = GISDocument(latitude=40.75, longitude=-73.95, zoom=12)
doc.add_raster_layer(url="https://basemap.nationalmap.gov/arcgis/rest/services/USGSTopo/MapServer/tile/{z}/{y}/{x}")
doc.add_openeo_tile_layer(result)
doc